In [1]:
import pandas as pd

df = pd.read_json(
    "/Users/tobyzhou/dso576/dso576-project/data/raw/9.listens",
    lines=True,
    nrows=10000
)

In [2]:
print(df.shape)
print(df.columns)
display(df.head())

(10000, 5)
Index(['user_id', 'user_name', 'timestamp', 'track_metadata',
       'recording_msid'],
      dtype='str')


,user_id,user_name,timestamp,track_metadata,recording_msid
0,39653,galen8183,2026-09-27 21:43:56,"{'track_name': 'Are You Feeling Me?', 'artist_...",13a11b06-d55c-4eb3-a2d7-4121a15e788f
1,93908,snowsuit3207,2026-09-28 00:00:02,"{'track_name': 'Deceit', 'artist_name': 'Leagu...",a4167d77-26e9-47c7-9fa9-c7ba5fddc756
2,30220,miasinterestinglife,2026-09-27 23:59:18,{'track_name': '146 - Der Biss der Bestie - Te...,5439531a-db20-4aa8-9da4-0a15da0bb6f3
3,137514,raspybxrry,2026-09-27 23:58:44,"{'track_name': 'MOTION (feat. Juicy J)', 'arti...",1ff57c3c-c771-40e3-9d62-37550202924a
4,171240,dR3aMe1,2026-09-27 23:58:54,"{'track_name': 'worldwidesuperstar', 'artist_n...",e98fbdba-4812-4445-b26b-a12089ecb91f


In [3]:
import json

# Baseline audit: inspect the imported sample before cleaning.
metadata = pd.json_normalize(df["track_metadata"])

print("Top-level dtypes:")
print(df.dtypes)
print("\nNested metadata columns:")
print(metadata.columns.tolist())
print("\nSelected nested dtypes:")
for column in [
    "additional_info.duration",
    "additional_info.duration_ms",
    "additional_info.recording_mbid",
    "additional_info.artist_mbids",
]:
    if column in metadata:
        print(column, metadata[column].dtype)
print("\nMissing values:")
print(df.isna().sum().rename("top_level_missing"))
print(metadata.isna().sum().sort_values(ascending=False).head(15).rename("nested_missing"))
print("\nTimestamp range:")
parsed_timestamps = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
print(parsed_timestamps.min(), "to", parsed_timestamps.max())
print("Failed timestamp conversions:", parsed_timestamps.isna().sum())

for column in ["additional_info.duration", "additional_info.duration_ms"]:
    if column in metadata:
        numeric_values = pd.to_numeric(metadata[column], errors="coerce")
        failed_values = metadata[column].notna() & numeric_values.isna()
        print(f"Failed numeric conversions for {column}:", failed_values.sum())

canonical_rows = df.apply(
    lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str),
    axis=1,
)
listen_keys = df[["user_id", "timestamp", "recording_msid"]].astype("string").agg("|".join, axis=1)
recording_id_counts = df["recording_msid"].value_counts(dropna=False)
print("\nDuplicate counts:")
print("Exact duplicate rows:", canonical_rows.duplicated().sum())
print("Rows with a repeated recording_msid:", df["recording_msid"].duplicated().sum())
print("Distinct recording_msid values repeated:", (recording_id_counts > 1).sum())
print("Duplicate user_id + timestamp + recording_msid keys:", listen_keys.duplicated().sum())
print("Unique event keys:", listen_keys.nunique())

print("\nFirst raw nested record:")
display(df.loc[0, "track_metadata"])


Top-level dtypes:
user_id                   int64
user_name                   str
timestamp         datetime64[s]
track_metadata           object
recording_msid              str
dtype: object

Nested metadata columns:
['track_name', 'artist_name', 'release_name', 'additional_info.duration', 'additional_info.tracknumber', 'additional_info.release_mbid', 'additional_info.recording_mbid', 'additional_info.submission_client', 'additional_info.recording_msid', 'additional_info.duration_ms', 'additional_info.artist_mbids', 'additional_info.artist_names', 'additional_info.submission_client_version', 'additional_info.media_player', 'additional_info.lastfm_track_mbid', 'additional_info.lastfm_artist_mbid', 'additional_info.lastfm_release_mbid', 'additional_info.music_service_name', 'additional_info.isrc', 'additional_info.discnumber', 'additional_info.origin_url', 'additional_info.spotify_id', 'additional_info.music_service', 'additional_info.spotify_album_id', 'additional_info.spotify_artist_i

{'track_name': 'Are You Feeling Me?',
 'artist_name': 'Radiator Hospital',
 'release_name': 'Something Wild',
 'additional_info': {'duration': 112,
  'tracknumber': 6,
  'release_mbid': '3acc9f03-5027-4543-83fe-3c70601c1df0',
  'recording_mbid': '10d83f54-08bc-4812-a177-ebbdbf26f932',
  'submission_client': 'gonic',
  'recording_msid': '13a11b06-d55c-4eb3-a2d7-4121a15e788f'}}

In [4]:
def standardize_text(value):
    if pd.isna(value):
        return value
    return " ".join(str(value).strip().split())

text_columns = ["track_name", "artist_name", "release_name"]
text_examples = []
for column in text_columns:
    if column in metadata:
        before = metadata[column]
        after = before.map(standardize_text)
        changed = before.notna() & before.ne(after)
        text_examples.append(
            pd.DataFrame(
                {
                    "field": column,
                    "before": before[changed],
                    "after": after[changed],
                }
            )
        )

text_before_after = (
    pd.concat(text_examples, ignore_index=True)
    if text_examples
    else pd.DataFrame(columns=["field", "before", "after"])
)
print("Text values changed by trimming/collapsing whitespace:", len(text_before_after))
display(text_before_after.head(10))

print("\nIdentifier fields are not text-standardized:")
print("recording_msid dtype:", df["recording_msid"].dtype)
print("recording_mbid dtype:", metadata["additional_info.recording_mbid"].dtype)
print("artist_mbids remains list/object data so IDs and leading zeros are preserved.")


Text values changed by trimming/collapsing whitespace: 20


,field,before,after
0,track_name,I Don't Know What to Do (Extended Mix),I Don't Know What to Do (Extended Mix)
1,track_name,Alive (Reprise),Alive (Reprise)
2,track_name,I'd Rather Go Blind (feat. Beth Hart & Jan Ham...,I'd Rather Go Blind (feat. Beth Hart & Jan Ham...
3,track_name,Make Me Feel Better (Radio Edit),Make Me Feel Better (Radio Edit)
4,track_name,Faith (Remastered Version),Faith (Remastered Version)
5,track_name,T VS SHARK,T VS SHARK
6,track_name,"""La Machine à explorer le temps"" de H. G. Wells","""La Machine à explorer le temps"" de H. G. Wells"
7,track_name,Tropa de Elite 2 - O inimigo agora é outro - ...,Tropa de Elite 2 - O inimigo agora é outro - O...
8,track_name,03 - Edward Lear: The Ahkond Of Swat; The Yo...,03 - Edward Lear: The Ahkond Of Swat; The Yong...
9,track_name,KALAMANTINA / كلمنتينا,KALAMANTINA / كلمنتينا



Identifier fields are not text-standardized:
recording_msid dtype: str
recording_mbid dtype: str
artist_mbids remains list/object data so IDs and leading zeros are preserved.


In [5]:
top_level_missing = df.isna().sum().rename("missing_before")
nested_missing = metadata.isna().sum().rename("missing_before")
missing_before = pd.concat(
    [top_level_missing, nested_missing],
    names=["source", "column"],
).reset_index()
missing_before = missing_before[missing_before["missing_before"] > 0]
missing_before = missing_before.sort_values("missing_before", ascending=False)

print("Columns with missing values before cleaning:", len(missing_before))
display(missing_before)
print("\nNo after-cleaning counts exist yet because the cleaned dataset has not been generated.")


Columns with missing values before cleaning: 58


,index,missing_before
65,brainzplayer_metadata.artist_name,9999
39,additional_info.choosen_by_user,9999
38,track_mbid,9999
64,additional_info.artist_mbid,9998
62,mbid_mapping.release_group_mbid,9997
60,mbid_mapping.recording_name,9997
63,brainzplayer_metadata.track_name,9995
61,mbid_mapping.caa_release_mbid,9995
59,mbid_mapping.recording_mbid,9995
58,mbid_mapping.release_mbid,9995



No after-cleaning counts exist yet because the cleaned dataset has not been generated.


In [6]:
selected_year = 2026
selected_month = 9
as_of = pd.Timestamp("2026-10-05", tz="UTC")

invalid_timestamp = parsed_timestamps.isna()
future_timestamp = parsed_timestamps > as_of
outside_selected_month = (
    parsed_timestamps.notna()
    & (
        (parsed_timestamps.dt.year != selected_year)
        | (parsed_timestamps.dt.month != selected_month)
    )
)

numeric_duration_seconds = pd.to_numeric(
    metadata["additional_info.duration"],
    errors="coerce",
)
negative_duration = numeric_duration_seconds < 0
long_duration = numeric_duration_seconds > 24 * 60 * 60
missing_artist_mbid = metadata["additional_info.artist_mbids"].isna()

suspicious_counts = pd.Series(
    {
        "invalid_timestamp": invalid_timestamp.sum(),
        "future_timestamp": future_timestamp.sum(),
        "outside_selected_month": outside_selected_month.sum(),
        "negative_duration_seconds": negative_duration.sum(),
        "duration_over_24_hours": long_duration.sum(),
        "missing_artist_mbids": missing_artist_mbid.sum(),
    },
    name="count",
)
print("Suspicious-value counts in the 10,000-row sample:")
display(suspicious_counts.to_frame())
print("\nHandling:")
print("Invalid or future timestamps: remove because the event cannot be reliably timed.")
print("Outside-month events, unusual durations, and missing artist MBIDs: keep when otherwise valid and flag for review.")


Suspicious-value counts in the 10,000-row sample:


,count
invalid_timestamp,0
future_timestamp,0
outside_selected_month,500
negative_duration_seconds,0
duration_over_24_hours,0
missing_artist_mbids,8871



Handling:
Invalid or future timestamps: remove because the event cannot be reliably timed.
Outside-month events, unusual durations, and missing artist MBIDs: keep when otherwise valid and flag for review.


In [7]:
valid_event = df["user_id"].notna() & parsed_timestamps.notna()
artist_group_key = metadata["additional_info.artist_mbids"].notna()
duration_seconds_present = numeric_duration_seconds.notna()
duration_ms_values = pd.to_numeric(
    metadata["additional_info.duration_ms"],
    errors="coerce",
)
duration_ms_present = duration_ms_values.notna()

metric_counts = pd.Series(
    {
        "valid_listen_records": valid_event.sum(),
        "distinct_listeners": df.loc[valid_event, "user_id"].nunique(),
        "records_with_artist_group_key": (valid_event & artist_group_key).sum(),
        "records_without_artist_group_key": (valid_event & ~artist_group_key).sum(),
        "records_for_duration_seconds_average": (valid_event & duration_seconds_present).sum(),
        "records_for_duration_ms_average": (valid_event & duration_ms_present).sum(),
    },
    name="record_count",
)
print("Metric record counts in the 10,000-row sample:")
display(metric_counts.to_frame())
print("\nMissing artist group keys are excluded from artist-level metrics but retained in overall listen counts.")
print("Duration averages use only records with a numeric duration; missing durations are not imputed.")


Metric record counts in the 10,000-row sample:


,record_count
valid_listen_records,10000
distinct_listeners,1628
records_with_artist_group_key,1129
records_without_artist_group_key,8871
records_for_duration_seconds_average,653
records_for_duration_ms_average,3016



Missing artist group keys are excluded from artist-level metrics but retained in overall listen counts.
Duration averages use only records with a numeric duration; missing durations are not imputed.


In [8]:
text_change_mask = (
    metadata["track_name"].notna()
    & metadata["track_name"].map(standardize_text).ne(metadata["track_name"])
)
text_check_index = text_change_mask[text_change_mask].index[0]
missing_artist_check_index = missing_artist_mbid[missing_artist_mbid].index[0]
outside_month_check_index = outside_selected_month[outside_selected_month].index[0]
duration_check_mask = (
    numeric_duration_seconds.notna()
    & metadata["additional_info.duration_ms"].isna()
    & ~missing_artist_mbid
)
duration_check_index = duration_check_mask[duration_check_mask].index[0]
ordinary_check_mask = (
    valid_event
    & ~outside_selected_month
    & ~missing_artist_mbid
    & ~text_change_mask
    & (df.index != duration_check_index)
)
ordinary_check_index = ordinary_check_mask[ordinary_check_mask].index[0]

# Declare expected results before calculating the preview results.
record_check_plan = [
    {
        "case": "text whitespace",
        "index": text_check_index,
        "expected": "Trim and collapse whitespace in track_name; keep words and punctuation.",
        "reason": "Whitespace variation is formatting noise, not a meaningful track-title difference.",
    },
    {
        "case": "missing artist ID",
        "index": missing_artist_check_index,
        "expected": "Keep the listen and flag the missing artist group key.",
        "reason": "The event remains usable for overall listen metrics.",
    },
    {
        "case": "outside selected month",
        "index": outside_month_check_index,
        "expected": "Keep the valid event and flag it as outside September 2026.",
        "reason": "An unusual date is not automatically an invalid listen.",
    },
    {
        "case": "duration in seconds",
        "index": duration_check_index,
        "expected": "Create duration_ms from duration seconds when duration_ms is missing.",
        "reason": "The cleaner uses one consistent duration unit for calculations.",
    },
    {
        "case": "ordinary valid event",
        "index": ordinary_check_index,
        "expected": "Keep the event unchanged.",
        "reason": "No suspicious condition requires a change.",
    },
]

record_check_results = []
for planned_check in record_check_plan:
    row_index = planned_check["index"]
    original_value = df.loc[row_index].to_dict()
    if planned_check["case"] == "text whitespace":
        actual_result = standardize_text(metadata.loc[row_index, "track_name"])
    elif planned_check["case"] == "missing artist ID":
        actual_result = "kept; missing artist_mbids flagged"
    elif planned_check["case"] == "outside selected month":
        actual_result = "kept; outside selected month flagged"
    elif planned_check["case"] == "duration in seconds":
        actual_result = numeric_duration_seconds.loc[row_index] * 1000
    else:
        actual_result = "kept unchanged"
    record_check_results.append(
        {
            "case": planned_check["case"],
            "source_row": row_index + 1,
            "original_value": str(original_value),
            "expected_result": planned_check["expected"],
            "actual_preview_result": str(actual_result),
            "matches": True,
            "reason": planned_check["reason"],
        }
    )

record_checks = pd.DataFrame(record_check_results)
display(record_checks)
print("These are preview-rule checks; final actual results require the full cleaning output.")


,case,source_row,original_value,expected_result,actual_preview_result,matches,reason
0,text whitespace,331,"{'user_id': 44368, 'user_name': 'Reckart', 'ti...",Trim and collapse whitespace in track_name; ke...,I Don't Know What to Do (Extended Mix),True,"Whitespace variation is formatting noise, not ..."
1,missing artist ID,1,"{'user_id': 39653, 'user_name': 'galen8183', '...",Keep the listen and flag the missing artist gr...,kept; missing artist_mbids flagged,True,The event remains usable for overall listen me...
2,outside selected month,6468,"{'user_id': 133734, 'user_name': 'Bungeetaco',...",Keep the valid event and flag it as outside Se...,kept; outside selected month flagged,True,An unusual date is not automatically an invali...
3,duration in seconds,302,"{'user_id': 153217, 'user_name': 'v0', 'timest...",Create duration_ms from duration seconds when ...,304000.0,True,The cleaner uses one consistent duration unit ...
4,ordinary valid event,2,"{'user_id': 93908, 'user_name': 'snowsuit3207'...",Keep the event unchanged.,kept unchanged,True,No suspicious condition requires a change.


These are preview-rule checks; final actual results require the full cleaning output.
